# 02 — Caching, Partitioning & Broadcast Join

In [1]:
import os, time
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *

GLUTEN_ENABLED = os.environ.get('GLUTEN_ENABLED', 'false').lower() == 'true'
MASTER   = 'spark://spark-master:7077'
DATA_DIR = '/workspace/data'

spark = (
    SparkSession.builder
    .appName('spark-notebook')
    .master(MASTER)
    .config('spark.executor.memory', '2g')
    .config('spark.driver.memory',   '1g')
    .config('spark.executor.cores',  '2')
    .getOrCreate()
)
spark.sparkContext.setLogLevel('WARN')
mode = 'Gluten/Velox' if GLUTEN_ENABLED else 'Vanilla'
print(f'Spark {spark.version}  |  Mode: {mode}')
spark.range(3).show()


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/08 06:14:42 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
W20260408 06:14:44.496651  1205 MemoryArbitrator.cpp:84] Query memory capacity[288.00MB] is set for NOOP arbitrator which has no capacity enforcement
26/04/08 06:14:45 WARN SparkShimProvider: Spark runtime version 4.0.2 is not matched with Gluten's fully tested version 4.0.1


Spark 4.0.2  |  Mode: Gluten/Velox


26/04/08 06:14:47 WARN GlutenFallbackReporter: Validation failed for plan: Range[QueryId=0], due to: does not support ansi mode
26/04/08 06:14:47 WARN GlutenFallbackReporter: Validation failed for plan: Project[QueryId=0], due to: does not support ansi mode
[Stage 0:>                                                          (0 + 4) / 4]

+---+
| id|
+---+
|  0|
|  1|
|  2|
+---+



## Caching & Persist

In [2]:
from pyspark import StorageLevel
big_df = spark.range(1_000_000).withColumn('val', F.rand()).withColumn('grp', (F.col('id') % 100).cast('string'))

t0 = time.time(); big_df.count(); print(f'Cold: {time.time()-t0:.2f}s')
big_df.persist(StorageLevel.MEMORY_AND_DISK)
t0 = time.time(); big_df.count(); print(f'Warm: {time.time()-t0:.2f}s')
big_df.unpersist()


26/04/08 06:14:52 WARN GlutenFallbackReporter: Validation failed for plan: AdaptiveSparkPlan[QueryId=1], due to: does not support ansi mode
26/04/08 06:14:52 WARN GlutenFallbackReporter: Validation failed for plan: Range[QueryId=1], due to: does not support ansi mode
26/04/08 06:14:52 WARN GlutenFallbackReporter: Validation failed for plan: Project[QueryId=1], due to: does not support ansi mode
26/04/08 06:14:52 WARN GlutenFallbackReporter: Validation failed for plan: HashAggregate[QueryId=1], due to: does not support ansi mode
26/04/08 06:14:52 WARN GlutenFallbackReporter: Validation failed for plan: Exchange[QueryId=1], due to: does not support ansi mode
26/04/08 06:14:52 WARN GlutenFallbackReporter: Validation failed for plan: ShuffleQueryStage[QueryId=1], due to: does not support ansi mode
26/04/08 06:14:52 WARN GlutenFallbackReporter: Validation failed for plan: HashAggregate[QueryId=1], due to: does not support ansi mode


Cold: 0.57s


26/04/08 06:14:53 WARN GlutenFallbackReporter: Validation failed for plan: Range, due to: does not support ansi mode
26/04/08 06:14:53 WARN GlutenFallbackReporter: Validation failed for plan: Project, due to: does not support ansi mode
26/04/08 06:14:53 WARN GlutenFallbackReporter: Validation failed for plan: Project, due to: does not support ansi mode
26/04/08 06:14:53 WARN GlutenFallbackReporter: Validation failed for plan: AdaptiveSparkPlan[QueryId=2], due to: does not support ansi mode
26/04/08 06:14:53 WARN GlutenFallbackReporter: Validation failed for plan: TableCacheQueryStage[QueryId=2], due to: does not support ansi mode
26/04/08 06:14:53 WARN GlutenFallbackReporter: Validation failed for plan: HashAggregate[QueryId=2], due to: does not support ansi mode
26/04/08 06:14:53 WARN GlutenFallbackReporter: Validation failed for plan: Exchange[QueryId=2], due to: does not support ansi mode


Warm: 0.95s


26/04/08 06:14:53 WARN GlutenFallbackReporter: Validation failed for plan: ShuffleQueryStage[QueryId=2], due to: does not support ansi mode
26/04/08 06:14:53 WARN GlutenFallbackReporter: Validation failed for plan: HashAggregate[QueryId=2], due to: does not support ansi mode


DataFrame[id: bigint, val: double, grp: string]

## Partitioning

In [3]:
df = spark.range(500_000).withColumn('group', (F.col('id') % 5).cast('string'))
print('Default partitions:', df.rdd.getNumPartitions())

df_rep = df.repartition(10, 'group')
print('After repartition:', df_rep.rdd.getNumPartitions())

df_coal = df.coalesce(2)
print('After coalesce:', df_coal.rdd.getNumPartitions())


26/04/08 06:15:21 WARN GlutenFallbackReporter: Validation failed for plan: Range, due to: does not support ansi mode
26/04/08 06:15:21 WARN GlutenFallbackReporter: Validation failed for plan: Project, due to: does not support ansi mode
26/04/08 06:15:21 WARN GlutenFallbackReporter: Validation failed for plan: AdaptiveSparkPlan, due to: does not support ansi mode
26/04/08 06:15:21 WARN GlutenFallbackReporter: Validation failed for plan: Range, due to: does not support ansi mode
26/04/08 06:15:21 WARN GlutenFallbackReporter: Validation failed for plan: Project, due to: does not support ansi mode
26/04/08 06:15:21 WARN GlutenFallbackReporter: Validation failed for plan: Exchange, due to: does not support ansi mode


Default partitions: 4
After repartition: 10
After coalesce: 2


26/04/08 06:15:21 WARN GlutenFallbackReporter: Validation failed for plan: ShuffleQueryStage, due to: does not support ansi mode
26/04/08 06:15:21 WARN GlutenFallbackReporter: Validation failed for plan: Range, due to: does not support ansi mode
26/04/08 06:15:21 WARN GlutenFallbackReporter: Validation failed for plan: Project, due to: does not support ansi mode
26/04/08 06:15:21 WARN GlutenFallbackReporter: Validation failed for plan: Coalesce, due to: does not support ansi mode


## Broadcast Join

In [4]:
transactions = spark.range(500_000).withColumn('country_code', (F.col('id') % 4).cast('string'))
countries = spark.createDataFrame([('0','US'),('1','UK'),('2','DE'),('3','FR')], ['country_code','country_name'])

result = transactions.join(F.broadcast(countries), 'country_code')
t0 = time.time()
print('Count:', result.count(), f'  time: {time.time()-t0:.2f}s')
result.groupBy('country_name').count().show()


26/04/08 06:15:24 WARN GlutenFallbackReporter: Validation failed for plan: AdaptiveSparkPlan[QueryId=3], due to: does not support ansi mode
26/04/08 06:15:24 WARN GlutenFallbackReporter: Validation failed for plan: Scan ExistingRDD[QueryId=3], due to: does not support ansi mode
26/04/08 06:15:24 WARN GlutenFallbackReporter: Validation failed for plan: Filter[QueryId=3], due to: does not support ansi mode
26/04/08 06:15:24 WARN GlutenFallbackReporter: Validation failed for plan: Project[QueryId=3], due to: does not support ansi mode
26/04/08 06:15:24 WARN GlutenFallbackReporter: Validation failed for plan: BroadcastExchange[QueryId=3], due to: does not support ansi mode
26/04/08 06:15:25 WARN GlutenFallbackReporter: Validation failed for plan: Range[QueryId=3], due to: does not support ansi mode
26/04/08 06:15:25 WARN GlutenFallbackReporter: Validation failed for plan: Filter[QueryId=3], due to: does not support ansi mode
26/04/08 06:15:25 WARN GlutenFallbackReporter: Validation failed 

Count: 500000   time: 1.25s


26/04/08 06:15:26 WARN GlutenFallbackReporter: Validation failed for plan: Range[QueryId=4], due to: does not support ansi mode
26/04/08 06:15:26 WARN GlutenFallbackReporter: Validation failed for plan: Filter[QueryId=4], due to: does not support ansi mode
26/04/08 06:15:26 WARN GlutenFallbackReporter: Validation failed for plan: Project[QueryId=4], due to: does not support ansi mode
26/04/08 06:15:26 WARN GlutenFallbackReporter: Validation failed for plan: BroadcastQueryStage[QueryId=4], due to: does not support ansi mode
26/04/08 06:15:26 WARN GlutenFallbackReporter: Validation failed for plan: BroadcastHashJoin[QueryId=4], due to: does not support ansi mode
26/04/08 06:15:26 WARN GlutenFallbackReporter: Validation failed for plan: Project[QueryId=4], due to: does not support ansi mode
26/04/08 06:15:26 WARN GlutenFallbackReporter: Validation failed for plan: HashAggregate[QueryId=4], due to: does not support ansi mode
26/04/08 06:15:26 WARN GlutenFallbackReporter: Validation failed 

+------------+------+
|country_name| count|
+------------+------+
|          DE|125000|
|          US|125000|
|          UK|125000|
|          FR|125000|
+------------+------+



26/04/08 06:15:26 WARN GlutenFallbackReporter: Validation failed for plan: ShuffleQueryStage[QueryId=4], due to: does not support ansi mode
26/04/08 06:15:26 WARN GlutenFallbackReporter: Validation failed for plan: AQEShuffleRead[QueryId=4], due to: does not support ansi mode
26/04/08 06:15:26 WARN GlutenFallbackReporter: Validation failed for plan: HashAggregate[QueryId=4], due to: does not support ansi mode
